# 🤖 Fase 4: Modelado Predictivo & Machine Learning (Dynamic Pricing)
## Proyecto de Ciencia de Datos - Inside Airbnb NYC

---
### 🎯 Objetivos:
1. Establecer modelos baseline (Dummy Median, Ridge Regression).
2. Entrenar y comparar modelos avanzados basados en ensambles de árboles (Random Forest, LightGBM, XGBoost).
3. Evaluar métricas de error en dólares reales ($USD) y escala logarítmica (RMSE, MAE, MedAE, R²).
4. Optimizar hiperparámetros del mejor modelo.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.models import train_benchmark_models

# Carga de datos procesados
data_path = PROJECT_ROOT / "data" / "selected" / "airbnb" / "processed" / "nyc_listings_features.parquet"
df = pd.read_parquet(data_path)
print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas.")

---
## 1. Partición de Datos (Train / Test Split 80/20)

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Conjunto de Entrenamiento: {train_df.shape[0]:,} observaciones")
print(f"Conjunto de Prueba: {test_df.shape[0]:,} observaciones")

---
## 2. Entrenamiento y Comparación de Modelos de Regresión

In [ ]:
results_df, trained_models = train_benchmark_models(
    train_df,
    train_df['log_price'].values,
    train_df['price_usd'].values,
    test_df,
    test_df['log_price'].values,
    test_df['price_usd'].values
)

print("\n📊 TABLA COMPARATIVA DE RESULTADOS:")
results_df

---
## 3. Visualización de Predicciones vs Valores Reales

In [ ]:
best_model_name = "XGBoost"
best_preds_log = trained_models[best_model_name]["predictions_log"]
best_preds_usd = np.expm1(best_preds_log)
y_actual_usd = test_df['price_usd'].values

plt.figure(figsize=(10, 6))
plt.scatter(y_actual_usd, best_preds_usd, alpha=0.3, color='royalblue', s=20)
plt.plot([0, 1500], [0, 1500], '--', color='red', linewidth=2, label='Predicción Perfecta')
plt.title(f'Predicciones vs Valores Reales ($USD) - {best_model_name}', fontsize=14, fontweight='bold')
plt.xlabel('Precio Real ($USD)')
plt.ylabel('Precio Predicho ($USD)')
plt.xlim(0, 1500)
plt.ylim(0, 1500)
plt.legend()
plt.tight_layout()
plt.show()